# 03 — FOV46 Leiden sweep and manual resolution selection

This is the **mandatory checkpoint**. Run resolutions 0.1–1.0, inspect UMAP/markers/spatial structure, and do not proceed until a resolution is chosen.

In [ ]:
from pathlib import Path
import sys, os

# Notebook lives in PROJECT_ROOT/jupyter/.
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "jupyter" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.executable)
assert (PROJECT_ROOT / "scripts").exists(), "Run this notebook from PROJECT_ROOT/jupyter or PROJECT_ROOT."

## Production sweep

In [ ]:
%run ../scripts/04b_leiden_sweep_fov46.py

## Interactive summary

In [ ]:
import pandas as pd, scanpy as sc
from IPython.display import display
TAB=PROJECT_ROOT/"results/FOV46/tables"
summary=pd.read_csv(TAB/"leiden_resolution_sweep.csv")
display(summary)

## Load sweep object and compare selected candidate resolutions interactively

In [ ]:
a=sc.read_h5ad(PROJECT_ROOT/"results/FOV46/GSM9046088_FOV46_leiden_sweep.h5ad")
CANDIDATES=[0.4,0.5,0.6,0.7,0.8]  # edit
keys=[f"leiden_r{r:g}" for r in CANDIDATES]
sc.pl.umap(a,color=keys,ncols=3,frameon=True)

## Marker-panel dotplot for one candidate
Change `SELECTED_RESOLUTION` repeatedly while inspecting.

In [ ]:
from config.markers import MARKER_PANEL
SELECTED_RESOLUTION=0.6  # EDIT after inspection
key=f"leiden_r{SELECTED_RESOLUTION:g}"
present={ct:[g for g in genes if g in a.raw.var_names] for ct,genes in MARKER_PANEL.items()}
present={ct:g for ct,g in present.items() if g}
sc.pl.dotplot(a,var_names=present,groupby=key,use_raw=True,standard_scale="var",dendrogram=True)

## Top DE genes for the candidate resolution

In [ ]:
rank_key=f"rank_{key}_interactive"
sc.tl.rank_genes_groups(a,groupby=key,method="wilcoxon",use_raw=True,pts=True,key_added=rank_key)
sc.pl.rank_genes_groups(a,key=rank_key,n_genes=10,sharey=False)
markers=sc.get.rank_genes_groups_df(a,group=None,key=rank_key)
display(markers.groupby("group",observed=True).head(10))

## Record the manual choice
After inspecting the plots, edit `config/fov46_annotation.json`. Do **not** use this cell to invent cell-type labels.

In [ ]:
CFG=PROJECT_ROOT/"config/fov46_annotation.json"
print(CFG.read_text())